In [1]:
import json
import pandas as pd

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)

matrix = json.load(open("../data/raw/resorts.json", encoding="utf-8"))
resorts = pd.DataFrame(matrix[1:], columns=matrix[0])

print(resorts.shape)
print(resorts.columns.tolist())
resorts.head()

(32, 33)
['resort_id', 'name_ru', 'region', 'latitude', 'longitude', 'coordinate_quality', 'coordinate_source_title', 'coordinate_source_url', 'wikidata_id', 'piste_segments_6km', 'lift_lines_6km', 'lodging_objects_6km', 'food_objects_6km', 'transport_objects_6km', 'attraction_objects_6km', 'osm_radius_km', 'osm_overlap_warning', 'open_meteo_elevation_m', 'winter_mean_temperature_c', 'avg_snowfall_cm_per_season', 'avg_freezing_days_per_season', 'avg_snow_days_per_season', 'max_daily_snowfall_cm', 'weather_model', 'weather_period_start', 'weather_period_end', 'nearest_scheduled_airport', 'nearest_airport_iata', 'nearest_airport_distance_km', 'nearest_airport_municipality', 'completeness_pct', 'quality_flags', 'collected_at']


,resort_id,name_ru,region,latitude,longitude,coordinate_quality,coordinate_source_title,coordinate_source_url,wikidata_id,piste_segments_6km,lift_lines_6km,lodging_objects_6km,food_objects_6km,transport_objects_6km,attraction_objects_6km,osm_radius_km,osm_overlap_warning,open_meteo_elevation_m,winter_mean_temperature_c,avg_snowfall_cm_per_season,avg_freezing_days_per_season,avg_snow_days_per_season,max_daily_snowfall_cm,weather_model,weather_period_start,weather_period_end,nearest_scheduled_airport,nearest_airport_iata,nearest_airport_distance_km,nearest_airport_municipality,completeness_pct,quality_flags,collected_at
0,R001,Роза Хутор,Краснодарский край,43.646390,40.332220,nearby_feature_proxy,Роза Хутор,https://ru.wikipedia.org/wiki/%D0%A0%D0%BE%D0%...,Q1335490,334,64,86,107,45,42,6,"Possible overlap with Газпром Поляна (6.0 km),...",950,4.272848,390.082,35.2,92.6,47.67,ERA5,2021-11-01,2026-04-30,Sochi International Airport,AER,37.3,Sochi,1.0,coordinates are proxy; OSM radii overlap,46279.509709
1,R002,Красная Поляна,Краснодарский край,43.686110,40.260000,resort_or_complex_page,Красная Поляна (курорт),https://ru.wikipedia.org/wiki/%D0%9A%D1%80%D0%...,Q120478823,121,44,255,259,76,82,6,"Possible overlap with Газпром Поляна (1.6 km),...",513,7.114349,390.082,16.8,92.6,47.67,ERA5,2021-11-01,2026-04-30,Sochi International Airport,AER,35.9,Sochi,1.0,OSM radii overlap,46279.509709
2,R003,Газпром Поляна,Краснодарский край,43.684519,40.280172,resort_or_complex_page,Курорт Газпрома,https://ru.wikipedia.org/wiki/%D0%9A%D1%83%D1%...,Q4144556,240,63,223,243,77,77,6,"Possible overlap with Красная Поляна (1.6 km),...",546,6.900331,390.082,18.6,92.6,47.67,ERA5,2021-11-01,2026-04-30,Sochi International Airport,AER,36.9,Sochi,1.0,OSM radii overlap,46279.509709
3,R004,Шерегеш,Кемеровская область,52.953380,87.955480,resort_or_complex_page,Шерегеш (горнолыжный курорт),https://ru.wikipedia.org/wiki/%D0%A8%D0%B5%D1%...,Q21645716,142,23,233,57,54,4,6,None,695,-9.152539,243.922,151.6,93.2,21.07,ERA5,2021-11-01,2026-04-30,Spichenkovo Airport,NOZ,119.2,Novokuznetsk,1.0,None,46279.509709
4,R005,Архыз,Карачаево-Черкесская Республика,43.543330,41.181670,resort_or_complex_page,Архыз (курорт),https://ru.wikipedia.org/wiki/%D0%90%D1%80%D1%...,Q4127579,42,13,187,69,18,25,6,None,1655,0.470088,523.194,81.6,107.6,29.47,ERA5,2021-11-01,2026-04-30,Sochi International Airport,AER,99.4,Sochi,1.0,None,46279.509709


In [2]:
checks = {
    "строк": len(resorts),
    "уникальных resort_id": resorts["resort_id"].nunique(),
    "регионов": resorts["region"].nunique(),
    "широта в диапазоне": resorts["latitude"].between(-90, 90).all(),
    "долгота в диапазоне": resorts["longitude"].between(-180, 180).all(),
    "пропусков в координатах": int(resorts[["latitude", "longitude"]].isna().sum().sum()),
}
for k, v in checks.items():
    print(f"{k}: {v}")

print()
print(resorts["coordinate_quality"].value_counts())

строк: 32
уникальных resort_id: 32
регионов: 20
широта в диапазоне: True
долгота в диапазоне: True
пропусков в координатах: 0

coordinate_quality
manual_review_required    18
nearby_feature_proxy       8
resort_or_complex_page     6
Name: count, dtype: int64


In [3]:
review = resorts[["resort_id", "name_ru", "region", "latitude", "longitude", "coordinate_quality"]].copy()
review["source_name"] = review["name_ru"]       # исходное написание, не менять
review["статус_проверки"] = ""
review["ссылка_на_подтверждение"] = ""
review["комментарий"] = ""

order = {"manual_review_required": 0, "nearby_feature_proxy": 1, "resort_or_complex_page": 2}
review = review.sort_values("coordinate_quality", key=lambda s: s.map(order))

review.to_csv("../data/coordinate_review.csv", index=False, encoding="utf-8-sig")
review.head(20)

,resort_id,name_ru,region,latitude,longitude,coordinate_quality,source_name,статус_проверки,ссылка_на_подтверждение,комментарий
15,R016,Горный воздух,Сахалинская область,46.955000,142.78900,manual_review_required,Горный воздух,,,
16,R017,Холдоми,Хабаровский край,50.720000,136.41000,manual_review_required,Холдоми,,,
19,R020,Соболиная гора,Иркутская область,51.500000,104.15000,manual_review_required,Соболиная гора,,,
14,R015,КолаСпортЛэнд,Мурманская область,67.600000,33.66000,manual_review_required,КолаСпортЛэнд,,,
13,R014,Кукисвумчорр,Мурманская область,67.669000,33.72000,manual_review_required,Кукисвумчорр,,,
12,R013,Большой Вудъявр,Мурманская область,67.610000,33.69000,manual_review_required,Большой Вудъявр,,,
11,R012,Телецкий,Республика Алтай,51.771000,87.23000,manual_review_required,Телецкий,,,
10,R011,Манжерок,Республика Алтай,51.811000,85.80100,manual_review_required,Манжерок,,,
17,R018,Морозная гора,Камчатский край,53.183000,158.45100,manual_review_required,Морозная гора,,,
20,R021,Гора Белая,Свердловская область,57.660000,59.70000,manual_review_required,Гора Белая,,,


In [4]:
import math
from itertools import combinations

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0088
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dphi = p2 - p1
    dlmb = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dlmb/2)**2
    return 2 * R * math.asin(math.sqrt(a))

rows = []
for a, b in combinations(resorts.to_dict("records"), 2):
    d = haversine(a["latitude"], a["longitude"], b["latitude"], b["longitude"])
    if d < 12:
        rows.append((a["resort_id"], a["name_ru"], b["resort_id"], b["name_ru"], round(d, 1)))

overlap = pd.DataFrame(rows, columns=["id_a", "курорт_a", "id_b", "курорт_b", "расстояние_км"])
overlap.sort_values("расстояние_км")

,id_a,курорт_a,id_b,курорт_b,расстояние_км
3,R007,Эльбрус,R008,Чегет,1.4
2,R002,Красная Поляна,R003,Газпром Поляна,1.6
6,R013,Большой Вудъявр,R015,КолаСпортЛэнд,1.7
1,R001,Роза Хутор,R003,Газпром Поляна,6.0
5,R013,Большой Вудъявр,R014,Кукисвумчорр,6.7
0,R001,Роза Хутор,R002,Красная Поляна,7.3
7,R014,Кукисвумчорр,R015,КолаСпортЛэнд,8.1
4,R009,Мамисон,R010,Цей,11.5


In [5]:
import numpy as np
overlap["доля_пересечения_%"] = (
    2*6**2*np.arccos(overlap["расстояние_км"]/12)
    - overlap["расстояние_км"]/2*np.sqrt(4*6**2 - overlap["расстояние_км"]**2)
) / (np.pi*6**2) * 100
overlap.to_csv("../data/overlap_pairs.csv", index=False, encoding="utf-8-sig")

In [6]:
raw = pd.read_excel("../data/reference/coordinate_review.xlsx")
print(raw.columns.tolist())

['resort_id', 'name_ru', 'region', 'latitude', 'longitude', 'coordinate_quality', 'source_name', 'статус_проверки', 'ссылка_на_подтверждение', 'комментарий', 'lat_corrected', 'lon_corrected', 'operating_status', 'дата_проверки']


In [7]:
import re
import pandas as pd

raw = pd.read_excel("../data/reference/coordinate_review.xlsx")

def parse_corrected(row):
    nums = []
    for c in ("lat_corrected", "lon_corrected"):
        v = row[c]
        if pd.isna(v):
            continue
        if isinstance(v, (int, float)):
            nums.append(float(v))
            continue
        s = str(v).strip()
        if "." not in s:
            s = s.replace(",", ".")
        nums += [float(x) for x in re.findall(r"-?\d+\.\d+", s)]
    return nums if len(nums) == 2 else [None, None]

raw[["lat_corrected", "lon_corrected"]] = raw.apply(
    lambda r: pd.Series(parse_corrected(r)), axis=1
)
raw["lat_final"] = raw["lat_corrected"].fillna(raw["latitude"])
raw["lon_final"] = raw["lon_corrected"].fillna(raw["longitude"])
raw["shift_km"] = [
    haversine(a, b, c, d)
    for a, b, c, d in zip(raw.latitude, raw.longitude, raw.lat_final, raw.lon_final)
]

# ваши значения сохраняются, пустые клетки считаем «работает»
raw["operating_status"] = raw["operating_status"].fillna("operating")
print(raw["operating_status"].value_counts(), "\n")
print(raw["статус_проверки"].value_counts(), "\n")

# проверки
assert len(raw) == 32 and raw.resort_id.is_unique
assert raw.lat_final.between(41, 82).all() and raw.lon_final.between(19, 180).all()
assert raw.shift_km.max() < 50, "слишком большой сдвиг, проверьте опечатку"

# противоречия: «подтверждена», но точка сдвинута больше чем на 1.5 км
conflict = raw[(raw["статус_проверки"] == "подтверждена") & (raw.shift_km > 1.5)]
print("Противоречия статуса и сдвига:")
print(conflict[["resort_id", "name_ru", "статус_проверки", "shift_km"]].round(1).to_string(index=False))

out = raw[["resort_id", "name_ru", "статус_проверки", "lat_final", "lon_final",
           "shift_km", "operating_status", "ссылка_на_подтверждение",
           "дата_проверки", "комментарий"]]
out.to_csv("../data/reference/coordinate_overrides.csv", index=False, encoding="utf-8-sig")
out.sort_values("shift_km", ascending=False).head(10)

operating_status
operating                    30
closed                        1
closed_for_reconstruction     1
Name: count, dtype: int64 

статус_проверки
неточная        20
подтверждена    11
не найдена       1
Name: count, dtype: int64 

Противоречия статуса и сдвига:
Empty DataFrame
Columns: [resort_id, name_ru, статус_проверки, shift_km]
Index: []


,resort_id,name_ru,статус_проверки,lat_final,lon_final,shift_km,operating_status,ссылка_на_подтверждение,дата_проверки,комментарий
16,R022,Евразия,неточная,55.383554,59.397690,37.162715,operating,https://euroasia.su/,2026-10-15,"Координаты указывают на посёлок, сам комплекс ..."
25,R032,Гладенькая,неточная,52.945248,91.359350,17.421125,operating,https://ski-gladenkaya.ru/ski/grafik_glk,2026-10-15,"Точка в городе в 17,3 км от комплекса"
1,R017,Холдоми,неточная,50.740299,136.568441,11.377774,operating,https://holdomi.ru/,2026-10-15,точка в лесу
8,R018,Морозная гора,неточная,53.181029,158.284106,11.123376,operating,https://moroznaya.gosuslugi.ru/,2026-10-15,Координаты Елизово
12,R025,Завьялиха,неточная,54.792476,58.535725,5.742285,operating,https://zavyaliha.su/gornolyzhnye-trassy,2026-10-15,Координаты указывают на лес
14,R027,Мраткино,неточная,53.994344,58.446767,5.325855,operating,https://vk.ru/glk_mratkino,2026-10-15,Точка в 6 км от комплекса
6,R012,Телецкий,неточная,51.746424,87.293979,5.182512,operating,https://teletski.ru/,2026-10-15,Координаты указывают на село
10,R023,Аджигардак,неточная,54.978401,57.335129,4.159809,closed_for_reconstruction,https://www.tourister.ru/world/europe/russia/c...,2026-10-15,"Закрыт на реконструкцию, ссылка как подтверждение"
23,R007,Эльбрус,неточная,43.265992,42.477874,3.104360,operating,https://resort-elbrus.ru/,2026-10-15,"Точка в селе в 3,7 км от комплекса"
15,R028,Губаха,неточная,58.864050,57.578677,2.881929,operating,https://go-gu.ru/,2026-10-15,"Точка в 2,9 км от комплекса"


In [8]:
need = raw[(raw.shift_km > 3) & raw["комментарий"].isna()]
need[["resort_id", "name_ru", "shift_km"]].round(1).sort_values("shift_km", ascending=False)

,resort_id,name_ru,shift_km


In [9]:
import re

def stated_km(text):
    if pd.isna(text):
        return None
    m = re.search(r"(\d+(?:[.,]\d+)?)\s*(км|километр\w*|м|метр\w*)(?![а-яё])", str(text))
    if not m:
        return None
    v = float(m.group(1).replace(",", "."))
    return v if m.group(2).startswith("к") else v / 1000

raw["stated_km"] = raw["комментарий"].map(stated_km)
raw["diff_km"] = (raw["stated_km"] - raw["shift_km"]).abs()

print("Не заполнено:")
for col in ["дата_проверки", "operating_status", "ссылка_на_подтверждение"]:
    print(" ", col, raw.loc[raw[col].isna(), "name_ru"].tolist())

print("\nКомментарий не сходится со сдвигом (> 1 км):")
print(raw.loc[raw["diff_km"] > 1, ["name_ru", "stated_km", "shift_km"]].round(1).to_string(index=False))

Не заполнено:
  дата_проверки []
  operating_status []
  ссылка_на_подтверждение []

Комментарий не сходится со сдвигом (> 1 км):
Empty DataFrame
Columns: [name_ru, stated_km, shift_km]
Index: []


In [10]:
assert raw["дата_проверки"].notna().all(), "нет даты проверки у: " + str(raw.loc[raw["дата_проверки"].isna(), "name_ru"].tolist())